# 📡 Photométrie forcée — Fink/LSST

Ce notebook récupère et affiche la **photométrie forcée** (`/api/v1/fp`) pour les
objets les plus brillants d'un tag donné.

La photométrie forcée signifie une **mesure effectuée à une coordonnée fixe**
dans une image, indépendamment d'une détection au-dessus du seuil.
À partir de la 2ème détection, les mesures des nuits précédentes sont incluses —
ce qui permet de voir l'**évolution pré-détection** du transitoire.

Pour chaque objet :
- Courbe de lumière **combinée** : détections officielles (points pleins)
  + photométrie forcée (points creux) sur le même axe
- Vue en **flux PSF** et en **magnitude AB**
- Mise en évidence de la **date de première détection officielle**
- Tableau récapitulatif avec liens portail

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources` · `/api/v1/fp`

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
TAG          = "sn_near_galaxy_candidate"  # Tag source des objets
N_ALERTS     = 50           # Alertes à récupérer
N_OBJ_DISPLAY = 10          # Nb max d'objets à afficher (les plus brillants)
STARTDATE    = None
STOPDATE     = None
BASE_URL     = "https://api.lsst.fink-portal.org"

# Sélection des objets : 'brightest' | 'latest' | 'all'
SELECT_BY    = 'brightest'

SAVE_FIGS    = True
OUTPUT_DIR   = "fp_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.lines as mlines
from IPython.display import display as ipy_display, HTML
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.titlesize': 13, 'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE', 'g': '#0077BB', 'r': '#33AA77',
    'i': '#DDAA33', 'z': '#BB5500', 'y': '#AA0000',
}
MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')
ZP_AB     = 2.5 * np.log10(3631e9)

def mjd_to_datetime(s):
    return MJD_EPOCH + pd.to_timedelta(pd.to_numeric(s, errors='coerce'), unit='D')

def mjd_to_datestr(m):
    try: return (MJD_EPOCH + pd.to_timedelta(float(m), unit='D')).strftime('%Y-%m-%d')
    except: return str(m)

def flux_to_mag(flux, flux_e):
    flux, flux_e = np.asarray(flux, float), np.asarray(flux_e, float)
    v = flux > 0
    mag = np.full(flux.shape, np.nan)
    me  = np.full(flux.shape, np.nan)
    mag[v] = -2.5 * np.log10(flux[v]) + ZP_AB
    me[v]  = (2.5 / np.log(10)) * np.abs(flux_e[v] / flux[v])
    return mag, me

print("Imports OK ✓")

## 3. Récupération des alertes et sélection des objets

In [ ]:
# ── Alertes ───────────────────────────────────────────────────────────────────
payload = {"tag": TAG, "n": N_ALERTS, "output-format": "json"}
if STARTDATE: payload["startdate"] = STARTDATE
if STOPDATE:  payload["stopdate"]  = STOPDATE
r = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
r.raise_for_status()
df_tags = pd.DataFrame(r.json())
df_tags.columns = [c.replace('r:','').replace('f:','') for c in df_tags.columns]
id_col = next(c for c in df_tags.columns if 'diaObjectId' in c)
all_oids = df_tags[id_col].astype(str).unique().tolist()
print(f"✓ {len(df_tags)} alertes, {len(all_oids)} objets uniques")

# ── Sources (détections officielles) ─────────────────────────────────────────
print("\nRécupération des sources...")
sources_list = []
for chunk in tqdm([all_oids[i:i+20] for i in range(0, len(all_oids), 20)],
                  desc='Sources', unit='lot'):
    try:
        r2 = requests.post(f"{BASE_URL}/api/v1/sources",
                           json={"diaObjectId": ','.join(chunk),
                                 "columns": "r:diaObjectId,r:midpointMjdTai,r:band,r:psfFlux,r:psfFluxErr",
                                 "output-format": "json"}, timeout=120)
        r2.raise_for_status()
        if r2.json():
            df = pd.DataFrame(r2.json())
            df.columns = [c.replace('r:','') for c in df.columns]
            sources_list.append(df)
    except Exception as e:
        print(f"  ✗ {e}")

df_sources = pd.concat(sources_list, ignore_index=True) if sources_list else pd.DataFrame()
for col in ['midpointMjdTai','psfFlux','psfFluxErr']:
    df_sources[col] = pd.to_numeric(df_sources[col], errors='coerce')

# ── Sélection des objets à afficher ──────────────────────────────────────────
if SELECT_BY == 'brightest':
    flux_max = df_sources.groupby('diaObjectId')['psfFlux'].max().sort_values(ascending=False)
    selected_oids = flux_max.head(N_OBJ_DISPLAY).index.tolist()
elif SELECT_BY == 'latest':
    mjd_max = df_sources.groupby('diaObjectId')['midpointMjdTai'].max().sort_values(ascending=False)
    selected_oids = mjd_max.head(N_OBJ_DISPLAY).index.tolist()
else:
    selected_oids = all_oids[:N_OBJ_DISPLAY]

print(f"\n{len(selected_oids)} objet(s) sélectionnés pour l'affichage ({SELECT_BY}).")

## 4. Téléchargement de la photométrie forcée (`/api/v1/fp`)

In [ ]:
fp_data = {}   # {oid: DataFrame}

for oid in tqdm(selected_oids, desc='Forced photometry', unit='obj'):
    try:
        r3 = requests.post(
            f"{BASE_URL}/api/v1/fp",
            json={
                "diaObjectId"  : str(oid),
                "columns"      : "r:diaObjectId,r:midpointMjdTai,r:band,r:psfFlux,r:psfFluxErr",
                "output-format": "json",
            },
            timeout=60
        )
        r3.raise_for_status()
        if r3.json():
            df_fp = pd.DataFrame(r3.json())
            df_fp.columns = [c.replace('r:','') for c in df_fp.columns]
            for col in ['midpointMjdTai','psfFlux','psfFluxErr']:
                df_fp[col] = pd.to_numeric(df_fp[col], errors='coerce')
            fp_data[str(oid)] = df_fp
            print(f"  ✓ {oid} — {len(df_fp)} mesures forced phot")
        else:
            print(f"  ⚠️  {oid} — aucune mesure forced phot")
    except Exception as e:
        print(f"  ✗ {oid} — {e}")

print(f"\n✓ Forced photometry récupérée pour {len(fp_data)} objets.")

## 5. Courbes de lumière combinées — détections + photométrie forcée

**Points pleins** = détections officielles (`/api/v1/sources`)  
**Points creux** = photométrie forcée (`/api/v1/fp`)  
**Ligne verticale** = date de la première détection officielle

In [ ]:
for oid in selected_oids:
    oid_str = str(oid)

    # Données de détection
    lc = df_sources[df_sources['diaObjectId'].astype(str)==oid_str].copy()
    lc = lc.sort_values('midpointMjdTai')
    lc['date'] = mjd_to_datetime(lc['midpointMjdTai'])

    # Données forced phot
    fp = fp_data.get(oid_str, pd.DataFrame())
    if not fp.empty:
        fp = fp.sort_values('midpointMjdTai')
        fp['date'] = mjd_to_datetime(fp['midpointMjdTai'])

    # Date première détection officielle
    first_det_mjd  = lc['midpointMjdTai'].min()
    first_det_date = mjd_to_datetime(pd.Series([first_det_mjd])).iloc[0]
    first_det_str  = mjd_to_datestr(first_det_mjd)

    # Flux max → magnitude pic
    flux_pk  = lc['psfFlux'].max()
    mag_str  = f"{-2.5*np.log10(flux_pk)+ZP_AB:.2f}" if flux_pk > 0 else '—'
    n_fp_pts = len(fp) if not fp.empty else 0

    title = (f"diaObjectId : {oid_str}   |   "
             f"1ère détection : {first_det_str}   "
             f"mag pic : {mag_str}   "
             f"forced phot : {n_fp_pts} mesures")

    fig, (ax_f, ax_m) = plt.subplots(1, 2, figsize=(16, 5), layout='constrained')
    fig.suptitle(title, fontsize=11, fontweight='bold')

    bands_all = sorted(set(
        list(lc['band'].dropna().unique()) +
        (list(fp['band'].dropna().unique()) if not fp.empty else [])
    ))

    for band in bands_all:
        color = FILTER_COLORS.get(band, '#888')

        # ── Détections officielles (points pleins) ─────────────────────────
        sub_lc = lc[lc['band']==band].dropna(subset=['psfFlux','psfFluxErr'])
        if not sub_lc.empty:
            ax_f.errorbar(sub_lc['date'], sub_lc['psfFlux'], yerr=sub_lc['psfFluxErr'],
                          fmt='o', color=color, label=f"${band}$ det",
                          ms=6, capsize=3, lw=1.0, alpha=0.9, zorder=5)
            mag_v, me_v = flux_to_mag(sub_lc['psfFlux'].values, sub_lc['psfFluxErr'].values)
            det = ~np.isnan(mag_v)
            if det.any():
                ax_m.errorbar(sub_lc['date'].values[det], mag_v[det], yerr=me_v[det],
                              fmt='o', color=color, ms=6, capsize=3, lw=1.0,
                              alpha=0.9, zorder=5)

        # ── Photométrie forcée (points creux) ─────────────────────────────
        if not fp.empty:
            sub_fp = fp[fp['band']==band].dropna(subset=['psfFlux','psfFluxErr'])
            if not sub_fp.empty:
                ax_f.errorbar(sub_fp['date'], sub_fp['psfFlux'], yerr=sub_fp['psfFluxErr'],
                              fmt='o', color=color, mfc='white', mec=color,
                              label=f"${band}$ fp",
                              ms=5, capsize=2, lw=0.7, alpha=0.65, zorder=4)
                mag_fp, me_fp = flux_to_mag(sub_fp['psfFlux'].values, sub_fp['psfFluxErr'].values)
                det_fp = ~np.isnan(mag_fp)
                if det_fp.any():
                    ax_m.errorbar(sub_fp['date'].values[det_fp], mag_fp[det_fp],
                                  yerr=me_fp[det_fp],
                                  fmt='o', color=color, mfc='white', mec=color,
                                  ms=5, capsize=2, lw=0.7, alpha=0.65, zorder=4)

    # Ligne de première détection
    for ax_ in [ax_f, ax_m]:
        ax_.axvline(first_det_date, color='black', lw=1.2, ls='--', alpha=0.7,
                    label='1ère détection officielle')
        ax_.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax_.xaxis.get_majorticklabels(), rotation=30, ha='right')
        ax_.grid(True, alpha=0.2, linewidth=0.5)

    ax_f.axhline(0, color='gray', lw=0.6, ls=':', alpha=0.5)
    ax_f.set_ylabel('Flux PSF (nJy)')
    ax_m.invert_yaxis()
    ax_m.set_ylabel('Magnitude AB')

    # Légende commune simplifiée
    legend_elems = [
        mlines.Line2D([0],[0], marker='o', color='gray', ms=6, ls='None',
                      label='Détection officielle'),
        mlines.Line2D([0],[0], marker='o', color='gray', ms=5, ls='None',
                      mfc='white', mec='gray', label='Forced photometry'),
        mlines.Line2D([0],[0], color='black', lw=1.2, ls='--',
                      label='1ère détection'),
    ] + [
        mlines.Line2D([0],[0], marker='o', color=FILTER_COLORS.get(b,'#888'),
                      ms=6, ls='None', label=f"filtre {b}")
        for b in bands_all
    ]
    ax_f.legend(handles=legend_elems, fontsize=8, ncol=2, loc='upper left')

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/fp_{oid_str}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/fp_{oid_str}.png", bbox_inches='tight', dpi=150)

    plt.show()

    ipy_display(HTML(
        f'<div style="font-size:12px;margin:2px 0 14px 4px;">'
        f'🔗 <b>Portail Fink/LSST</b> : '
        f'<a href="https://lsst.fink-portal.org/{oid_str}" target="_blank">'
        f'https://lsst.fink-portal.org/{oid_str}</a></div>'
    ))
    print()

print("\n✅ Affichage terminé.")

## 6. Tableau récapitulatif

In [ ]:
rows = []
for oid in selected_oids:
    oid_str = str(oid)
    lc = df_sources[df_sources['diaObjectId'].astype(str)==oid_str]
    fp = fp_data.get(oid_str, pd.DataFrame())

    flux_pk = lc['psfFlux'].max()
    mag_pk  = round(-2.5*np.log10(flux_pk)+ZP_AB, 2) if flux_pk > 0 else np.nan

    # Nb de nuits forced phot avant la 1ère détection
    first_mjd = lc['midpointMjdTai'].min()
    n_fp_pre  = 0
    if not fp.empty:
        n_fp_pre = (fp['midpointMjdTai'] < first_mjd).sum()

    rows.append({
        'diaObjectId'        : oid_str,
        '1ère détection'     : mjd_to_datestr(first_mjd),
        'N détections'       : len(lc),
        'N forced phot'      : len(fp),
        'N fp pré-détection' : int(n_fp_pre),
        'flux_max (nJy)'     : round(flux_pk, 1),
        'mag_pic'            : mag_pk,
        'filtres'            : ', '.join(sorted(lc['band'].dropna().unique())),
    })

df_sum = pd.DataFrame(rows).sort_values('flux_max (nJy)', ascending=False)
df_sum['Portail Fink'] = df_sum['diaObjectId'].apply(
    lambda o: f'<a href="https://lsst.fink-portal.org/{o}" target="_blank">🔗 {o}</a>')

html = df_sum.to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse:collapse; font-size:12px; width:100%; }
  .fink-table th { background:#f0f0f0; padding:6px 10px;
                   border-bottom:2px solid #ccc; text-align:left; }
  .fink-table td { padding:4px 10px; border-bottom:1px solid #eee;
                   text-align:left; white-space:nowrap; }
  .fink-table tr:hover td { background:#fafafa; }
</style>
"""
ipy_display(HTML(style + html))